In [1]:
%%capture
!pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install flash-attn==2.8.0.post2 --no-build-isolation
!pip install evo2

In [2]:
import os

WORK_DIR = "/content"
CACHE_DIR = "/content/hf"

os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_DIR}"

In [3]:
import os
import torch
import numpy as np
from Bio import SeqIO
from tqdm import tqdm
from evo2 import Evo2

In [4]:
!nvidia-smi
print("CUDA available:", torch.cuda.is_available())

Sun May 31 19:20:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
#  Load Evo2 model
evo2_model = Evo2('evo2_7b')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
layer_name = 'blocks.28'

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


/usr/local/lib/python3.12/dist-packages/evo2/models.py:282: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(

100%|██████████| 32/32 [00:00<00:00, 158.93it/s]


Extra keys in state_dict: {'blocks.30.projections._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.23.projections._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.26.projections._extra_state', 'blocks.3.mixer.attn._extra_state', 'blocks.6.mixer.mixer.filter.t', 'blocks.31.mixer.dense._extra_state', 'blocks.9.mixer.mixer.filter.t', 'blocks.14.projections._extra_state', 'blocks.10.mixer.attn._extra_state', 'blocks.27.projections._extra_state', 'blocks.21.projections._extra_state', 'blocks.8.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.1.projections._extra_state', 'blocks.28.projections._extra_state', 'blocks.2.mixer.mixer.filter.t', 'blocks.19.projections._extra_state', 'blocks.9.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.4.projections._extra_state', 'blocks.24.mixer.attn._extra_state', 'blocks.31.mixer.attn._extra_state', 'blocks.12.projections._extra_state', 'blocks.16.projections._extra_state', 'blocks.10.mixer.d

In [6]:
#  Download input data from GCS

!mkdir -p /home/jupyter/data
!gsutil -m cp -r gs://rojo_project/vfdb/vfdb_context /home/jupyter/data/

INPUT_DIR = "/home/jupyter/data/vfdb_context"
print("Number of FASTA files:", len([f for f in os.listdir(INPUT_DIR) if f.endswith(".fasta")]))


Streaming output truncated to the last 5000 lines.
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443675.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443715.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443685.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443755.1.blast.tsv...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443795.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443775.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443775.1.blast.tsv...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443785.1.blast.tsv...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443735.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443765.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443825.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443705.1.fasta...
Copying gs://rojo_project/vfdb/vfdb_context/GCA_963443765.1.blast.tsv...
Copying gs://rojo_project/vfdb/vfdb_

In [7]:
#  Output directory for .npz files
OUTPUT_DIR = "/home/jupyter/data/evo2_embeddings_npz"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [11]:
#  Function: get mean‑pooled embedding for a single sequence
def get_sequence_embedding(sequence, model, layer_name, device):
    tokens = model.tokenizer.tokenize(sequence)
    input_ids = torch.tensor(tokens, dtype=torch.int).unsqueeze(0).to(device)

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[layer_name]
        )
    # embeddings[layer_name] shape: (1, L, D) -> squeeze to (L, D)
    per_base = embeddings[layer_name].squeeze(0).cpu()
    # mean over length → (D,)
    return per_base.float().mean(dim=0).numpy()


In [12]:
#  Process one genome FASTA: embed all windows, aggregate,
#  save as .npz
def process_genome_fasta(fasta_path, output_npz_path, model, layer_name, device):
    window_vectors = []

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq)
        if len(seq) == 0:
            continue

        try:
            vec = get_sequence_embedding(seq, model, layer_name, device)
            window_vectors.append(vec)
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM → skipping {record.id}")
                torch.cuda.empty_cache()
                continue
            else:
                raise e

    if not window_vectors:
        print(f"No valid windows in {fasta_path}, skipping")
        return

    # Aggregate: mean + max concatenation
    arr = np.stack(window_vectors, axis=0)   # (n_windows, D)
    mean_vec = arr.mean(axis=0)
    max_vec = arr.max(axis=0)
    genome_embedding = np.concatenate([mean_vec, max_vec])   # (2*D,)

    # Save as compressed numpy
    np.savez_compressed(output_npz_path, embedding=genome_embedding)


In [13]:
#  Main loop over all FASTA files
fasta_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".fasta")]

for fasta_file in tqdm(fasta_files):
    genome_stem = fasta_file.replace(".fasta", "")
    input_path = os.path.join(INPUT_DIR, fasta_file)
    output_path = os.path.join(OUTPUT_DIR, f"{genome_stem}.npz")

    if os.path.exists(output_path):
        continue

    process_genome_fasta(input_path, output_path, evo2_model, layer_name, device)


#  Upload results back to GCS
!gsutil -m cp -r /home/jupyter/data/evo2_embeddings_npz gs://rojo_project/

100%|██████████| 29920/29920 [13:18:42<00:00,  1.60s/it]


Streaming output truncated to the last 5000 lines.
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_034817505.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_963431655.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_045798365.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_035728605.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_040818175.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_050048435.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_053010365.1.npz [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/data/evo2_embeddings_npz/GCA_040257945.1.npz [Content-Type=application/octet-stream]...
Copying file:

In [14]:
#verification
sample_npz = os.path.join(OUTPUT_DIR, os.listdir(OUTPUT_DIR)[0])
data = np.load(sample_npz)
print(f"Sample file: {sample_npz}")
print(f"Embedding shape: {data['embedding'].shape}")   # Should be (2*D,)
print(f"Number of genomes processed: {len(os.listdir(OUTPUT_DIR))}")

Sample file: /home/jupyter/data/evo2_embeddings_npz/GCA_040673135.1.npz
Embedding shape: (8192,)
Number of genomes processed: 29920
